
# Practice 3 - Learning on Topological Data I: GNNs, expressivity, higher-order message-passing

This notebook retraces Session 3 in code, using two libraries:

1. **Learning on sets**: permutation invariance and equivariance (Deep Sets)
2. **PyTorch Geometric (PyG)**: graphs, message-passing, and a node-classification demo
3. **Expressivity**: the Weisfeiler-Leman test, by hand and through an untrained GNN
4. **TopoNetX**: extending message-passing to simplicial complexes (higher-order domains)

Cells marked **`# TODO`** are for you to fill in. 


## 0. Setup

In [ ]:
# Packages not preinstalled on Colab (safe to re-run / already-satisfied locally)
!pip install -q torch_geometric toponetx

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)


## 1. Learning on sets: permutation invariance and equivariance

We have seen that by stacking node features into a matrix $X=(x_1,\dots,x_n)^\top\in\mathbb{R}^{n\times k}$
we are already setting the *order* in which we visit the nodes. 
We can represent a permutation $\sigma\in S_n$ by a permutationmatrix $P_\sigma$, $(P_\sigma)_{ij}=1$ iff $\sigma(i)=j$, 
which acts on another matrix $A$ by multiplying it on the left $P_\sigma A$, and thus permuting the order of its rows according to $\sigma$. 
Recall that we want our architectures $f_\theta$ to not depend on this arbitrary choice:

- **graph-level tasks:** $f_\theta$ is **permutation invariant** if $f_\theta(PX)=f_\theta(X)$
- **node-level tasks:** $f_\theta$ is **permutation equivariant** if $f_\theta(PX)=Pf_\theta(X)$



In [ ]:
def permutation_matrix(perm):
    # perm[i] = sigma(i). Builds P with (P)_{i, perm[i]} = 1, so (P @ X)[i] = X[perm[i]].
    n = len(perm)
    P = torch.zeros(n, n)
    P[torch.arange(n), perm] = 1.0
    return P

# same permutation as the worked example on slide 21 (0-indexed): sigma = (1234 -> 3421)
perm = torch.tensor([2, 3, 1, 0])
P = permutation_matrix(perm)
X = torch.arange(4 * 3, dtype=torch.float32).reshape(4, 3)
print("X =\n", X)
print("P @ X =\n", P @ X, "\n(row i of P@X is row sigma(i) of X)")


### Deep Sets: a permutation-invariant model

We are going to implement the Deep Set architecture $$f(X)=\varphi\big(\sum_{i\in V}\psi(\mathbf{x}_i)\big),$$ with $\psi:\mathbb R^k\to\mathbb R^m$
applied to every row independently and $\varphi:\mathbb R^m\to\mathbb R^p$ applied once to the
pooled sum. The sum could be any permutation-invariant aggregator $\oplus$ (sum, average, maximum...).


In [ ]:
class DeepSets(nn.Module):
    # f(X) = varphi( sum_i psi(x_i) )  -- Zaheer et al. (2017).
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.psi = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.phi = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, out_dim))

    def forward(self, X):
        # TODO(1): pool psi(X) over the NODE dimension (dim=0) with a sum, then apply self.phi.
        pooled = ...
        return self.phi(pooled)

torch.manual_seed(0)
model = DeepSets(in_dim=3, hidden=8, out_dim=1)

X = torch.randn(6, 3)
outputs = [model(permutation_matrix(torch.randperm(6)) @ X).item() for _ in range(5)]
print("f(X)         =", model(X).item())
print("f(P_sigma X) =", outputs, "\n(all equal up to floating point -> permutation INVARIANT)")


### A permutation-equivariant layer

For node-level tasks we instead want $f_\theta(PX)=Pf_\theta(X)$. A simple such layer
(Zaheer et al., 2017) is  

$$f(X)_i = W x_i +  U \left(\dfrac{1}{V} \sum_{1\leq j \leq n} X[j, :] \right)\in \mathbb{R}^{k}$$
a per-node linear map plus a *shared* summary of the whole set 
(which does not depend on $i$, so it commutes with the permutation).


In [ ]:
class EquivariantLayer(nn.Module):
    # y_i = x_i W + (mean_j x_j) U -- permutation EQUIVARIANT.
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Linear(dim, dim, bias=False)
        self.U = nn.Linear(dim, dim, bias=False)

    def forward(self, X):
        # TODO(2): implement y = W(X) + U(mean over nodes of X), broadcast over all rows.
        return ...

torch.manual_seed(0)
layer = EquivariantLayer(X.shape[1])
P = permutation_matrix(torch.randperm(6))

lhs = layer(P @ X)      # f(PX)
rhs = P @ layer(X)      # P f(X)
print("max |f(PX) - P f(X)| =", (lhs - rhs).abs().max().item(), " (-> 0 means that the layer is equivariant)")


## 2. Learning on graphs: message-passing with PyTorch Geometric

Going beyon nodes, to add edges to the model we need to include the adjacency matrix $A$ in the picture.
We now aim to learn a function depending on the node features and the adjacencies of the graph $f_\theta(X,A)$, 
where permutation invariance/equivariance now being:
- Invariance:$f_\theta(PX,PAP^\top)=f_\theta(X,A)$ 
- Equivariance:$f_\theta(PX,PAP^\top)=Pf_\theta(X,A)$). 

We use the graph from the slides throughout the section.


In [ ]:
from torch_geometric.data import Data
from torch_geometric.utils import to_dense_adj

# our running example
edges = [(0,1),(0,2),(1,2),(1,6),(2,3),(3,4),(3,5),(3,7),(4,5),(6,7)]
edge_index = torch.tensor(edges + [(b, a) for a, b in edges], dtype=torch.long).t().contiguous()
x = torch.eye(8)   # one-hot node features, we could also use random features or a constant feature vector for each node

demo_data = Data(x=x, edge_index=edge_index) # both storing the node features and the adjacencies
print(demo_data)

# we can recover the adjacency matrix from the edge_index representation
print("A (recovered from edge_index) =\n", to_dense_adj(demo_data.edge_index)[0].int())

nx.draw(nx.Graph(edges), nx.spring_layout(nx.Graph(edges), seed=0),
        with_labels=True, node_color="black", font_color="white", node_size=450)
plt.title("our running graph"); plt.show()


### Message-passing, by hand

Before using a library layer, we will write the **convolutional** message-passing update ourselves:
$$\mathbf{h}_i=\phi\Big(\mathbf{x}_i,\ \bigoplus_{j\in\mathcal N_i} c_{ij}\,\psi(\mathbf{x}_j)\Big),$$
with $\psi(\mathbf{x}_j)=\mathbf{x}_jW$, $\oplus=\text{sum}$, and $c_{ij}=1/\deg(i)$.


In [ ]:
from torch_geometric.utils import degree

def manual_conv_layer(x, edge_index, W):
    src, dst = edge_index                       # a message flows src -> dst
    msg = x[src] @ W                             # psi(x_j) for every directed edge (j, i)
    out = torch.zeros(x.size(0), msg.size(1))
    # TODO(3): add `msg` into `out` at the destination nodes (`dst`), then divide by
    #          the degree of each node to get the mean aggregation c_ij = 1/deg(i).
    #          Hint: use the function out.index_add_(0, index, source)
    ...
    deg = degree(dst, x.size(0)).clamp(min=1).unsqueeze(1)
    return out / deg

torch.manual_seed(0)
W = torch.randn(8, 4)
h = manual_conv_layer(demo_data.x, demo_data.edge_index, W)
print("h.shape =", h.shape)
print(h)


### The library version: `GCNConv` and `GATConv`

PyG provides more thab 50 local functions to implement your GNN. `GCNConv` (Kipf and Welling, 2017) is the
**convolutional** message-passing layer from above; `GATConv` (Veličković et al., 2018) is the **attentional**
message-passing layer, replacing the fixed $c_{ij}$ with a learned attention score $\alpha(x_i,x_j)$.


In [ ]:
from torch_geometric.nn import GCNConv, GATConv

class GNN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, conv_layer=GCNConv):
        super().__init__()
        self.conv1 = conv_layer(in_dim, hidden)
        self.conv2 = conv_layer(hidden, out_dim)

    def forward(self, x, edge_index):
        # TODO(4): stack the two conv layers with a ReLU in between (like the MLPs of practice 1).
        ...

torch.manual_seed(0)
gcn = GNN(demo_data.num_features, 8, 4, conv_layer=GCNConv)
gat = GNN(demo_data.num_features, 8, 4, conv_layer=GATConv)
print("GCN output:\n", gcn(demo_data.x, demo_data.edge_index))
print("GAT output:\n", gat(demo_data.x, demo_data.edge_index))


## 3. Node classification: Zachary's Karate Club

The classic PyG "hello world": A social network of a karate club was studied by Wayne W. Zachary for a period of three years from 1970 to 1972 (Zachary, 1977). The network captures 34 members of a karate club, documenting links between pairs of members who interacted outside the club. During the study a conflict arose between the administrator "John A" and instructor "Mr. Hi" (pseudonyms), which led to the split of the club. The dataset that we will be working with records a split into 4 groups. 

We only know the community/faction of **4** members (`data.train_mask`) and train a GCN to predict everyone else's from the friendship graph alone. This is a  *semi-supervised* task, only possible because the GNN uses the graph structure $A$, not just node features.


In [ ]:
from torch_geometric.datasets import KarateClub
from torch_geometric.utils import to_networkx

dataset = KarateClub()
data = dataset[0]
print(data)
print("classes:", dataset.num_classes, " | labeled nodes:", int(data.train_mask.sum()))

G = to_networkx(data, to_undirected=True)
plt.figure(figsize=(5, 5))
nx.draw(G, nx.spring_layout(G, seed=0), node_color=data.y, cmap="Set2",
        with_labels=True, node_size=350)
plt.title("Karate Club"); plt.show()

In [ ]:
class KarateGCN(nn.Module):
    def __init__(self, num_features, num_classes, hidden=4, embed_dim=2):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, embed_dim)     # 2-D so we can plot it directly
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index).tanh()
        h = self.conv2(h, edge_index).tanh()
        h = self.conv3(h, edge_index).tanh()        # final node EMBEDDING
        return self.classifier(h), h

def visualize_embedding(h, color, title=""):
    h = h.detach().cpu().numpy()
    plt.figure(figsize=(4, 4))
    plt.scatter(h[:, 0], h[:, 1], c=color, cmap="Set2", s=140, edgecolors="k")
    plt.title(title); plt.show()

torch.manual_seed(0)
untrained = KarateGCN(dataset.num_features, dataset.num_classes)
_, h0 = untrained(data.x, data.edge_index)
visualize_embedding(h0, data.y, "untrained GCN -- structure alone already helps!")

In [ ]:
torch.manual_seed(0)
model = KarateGCN(dataset.num_features, dataset.num_classes)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

acc_history = []
for epoch in range(200):
    out, h = model(data.x, data.edge_index)
    loss = loss_fn(out[data.train_mask], data.y[data.train_mask])   # loss on the 4 LABELED nodes only

    # TODO(5): the standard training step -- zero_grad, backward, step
    ...

    acc = (out.argmax(dim=1) == data.y).float().mean().item()       # accuracy on ALL 34 nodes
    acc_history.append(acc)

print(f"trained on 4 labels only -> final accuracy on all 34 nodes: {acc_history[-1]:.3f}")
plt.plot(acc_history); plt.xlabel("epoch"); plt.ylabel("accuracy (34 nodes)")
plt.title("semi-supervised node classification"); plt.show()


## 4. Expressivity: the Weisfeiler-Leman test

We have seen in the lecture that, over discrete features, message-passing GNNs are **at most as powerful as the
1-WL test**, since an *untrained* GNN behaves like the WL algorithm. 
Let's implement the 1-WL color refinement algorithms ourselves: start every vertex with the same color, then
repeatedly recolor each vertex by hashing its own color together with the *sorted multiset* of its neighbors' colors, until the color partition stops changing.


In [ ]:
from collections import Counter

def wl_refine(G, iterations=4):
    # 1-dimensional Weisfeiler-Leman color refinement (slides 33-38).
    color = {v: 0 for v in G.nodes()}                 # initialization: one color for everyone
    for _ in range(iterations):
        new_color = {}
        for v in G.nodes():
            # TODO(6): compute the sorted tuple of NEIGHBOR colors and use the in-built hash() function to get a new color for v
            neighbor_colors = ...
            new_color[v] = ...
        uniq = {c: i for i, c in enumerate(sorted(set(new_color.values())))}
        color = {v: uniq[new_color[v]] for v in new_color}     # relabel to small integers
    return color

def color_histogram(color):
    return tuple(sorted(Counter(color.values()).values()))

# a pair WL CAN tell apart: same #nodes/#edges, different degree sequence
G_path, G_star = nx.path_graph(4), nx.star_graph(3)
for name, G in [("path P4", G_path), ("star S4", G_star)]:
    print(f"{name:10s} WL histogram: {color_histogram(wl_refine(G))}")

In [ ]:
# Plot the WL color histograms
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)

for ax, (name, G) in zip(axes, [("Path P4", G_path), ("Star S4", G_star)]):
    counts = Counter(wl_refine(G).values())
    ids = sorted(counts)

    ax.bar(
        ids, [counts[c] for c in ids],
        color=[plt.get_cmap("tab10")(c) for c in ids],
    )
    ax.set(title=name, xlabel="WL color", ylabel="Number of vertices")
    ax.set_xticks(ids)
    ax.set_yticks(range(5))

plt.tight_layout()
plt.show()


### A pair that fools WL (and every message-passing GNN)

A hexagon $C_6$ and two disjoint triangles $C_3\sqcup C_3$: both are 2-regular, so **every**
vertex always sees an identical local picture (two neighbors, same color). WL -- and any
purely local message-passing GNN -- can never refine past a single color, even though the
graphs are obviously not isomorphic (1 connected component vs. 2)!


In [ ]:
G_hex = nx.cycle_graph(6)
G_two_tri = nx.disjoint_union(nx.cycle_graph(3), nx.cycle_graph(3))

for name, G in [("hexagon C6", G_hex), ("two triangles", G_two_tri)]:
    print(f"{name:15s} WL histogram: {color_histogram(wl_refine(G))}")

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
nx.draw(G_hex, ax=ax[0], with_labels=True, node_color="#4C72B0", font_color="white")
nx.draw(G_two_tri, ax=ax[1], with_labels=True, node_color="#DD8452", font_color="white")
ax[0].set_title("C6"); ax[1].set_title("C3 + C3"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)

for ax, (name, G) in zip(axes, [("hexagon C6", G_hex), ("two triangles", G_two_tri)]):
    counts = Counter(wl_refine(G).values())
    ids = sorted(counts)

    ax.bar(
        ids, [counts[c] for c in ids],
        color=[plt.get_cmap("tab10")(c) for c in ids],
    )
    ax.set(title=name, xlabel="WL color", ylabel="Number of vertices")
    ax.set_xticks(ids)
    ax.set_yticks(range(7))

plt.tight_layout()
plt.show()


### Seeing the failure in a real (untrained) GNN

In the lecture, we have seen that when we have an injective aggregator in message-passing, then the embeddings that message-passing provides are as discriminative as the WL test (Xu et al., 2018). Based on this principle, they define the Graph Isomoprphism Netwrok (`GINConv` in PyG), which reaches maximum expressivity. 

To test this, we give both graphs the same
constant node features and run the *same untrained* GIN, then compare a graph-level
readout (pooling all the node features by summing them, thus obtaining a permutation-invariant representation for the graph).


In [ ]:
from torch_geometric.nn import GINConv
from torch_geometric.utils import from_networkx

def make_gin(in_dim=1, hidden=8, layers=3, seed=0):
    torch.manual_seed(seed)
    convs = nn.ModuleList()
    d = in_dim
    for _ in range(layers):
        mlp = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        convs.append(GINConv(mlp))          
        d = hidden
    return convs

def gin_readout(G, convs):
    ei = from_networkx(G).edge_index
    x = torch.ones(G.number_of_nodes(), 1)   # identical constant features everywhere
    for conv in convs:
        x = conv(x, ei).relu()
    return x.sum(dim=0)                       # graph-level readout by sum pooling

r_hex, r_tri = gin_readout(G_hex, make_gin()), gin_readout(G_two_tri, make_gin())
r_path, r_star = gin_readout(G_path, make_gin()), gin_readout(G_star, make_gin())

print(f"||readout(hexagon) - readout(two triangles)|| = {(r_hex - r_tri).norm().item():.2e}  (WL-indistinguishable -> SAME embedding)")
print(f"||readout(path)    - readout(star)||          = {(r_path - r_star).norm().item():.2e}  (WL-distinguishable   -> DIFFERENT embedding)")


## 5. Beyond graphs: higher-order message-passing with TopoNetX

We closed the lecture by extending WL (and message-passing) from graphs to **simplicial complexes**,
using exactly the boundary/coboundary/upper/lower adjacency machinery from Lecture 2:
$$L_k^{\text{down}}=B_k^\top B_k$, $L_k^{\text{up}}=B_{k+1}B_{k+1}^\top.$$ 
TopoNetX (part of the `pyt-team` **TopoX**) is a library that builds these matrices for us.

We reuse the complex from Problem Sheet 2, Exercise 2(c): two filled triangles $\{0,1,2\}$
and $\{1,2,3\}$ sharing the edge $\{1,2\}$, so that you can check your answers to that excercise!


In [ ]:
from toponetx.classes import SimplicialComplex

SC = SimplicialComplex([[0, 1, 2], [1, 2, 3]])
n0, n1, n2 = SC.shape
print(f"{n0} vertices, {n1} edges, {n2} triangles")
print("edges:   ", SC.skeleton(1))
print("triangles:", SC.skeleton(2))

In [ ]:
B1 = SC.incidence_matrix(rank=1).toarray()
B2 = SC.incidence_matrix(rank=2).toarray()
print("B1 =\n", B1.astype(int))
print("B2 =\n", B2.astype(int))
print("B1 @ B2 =\n", (B1 @ B2).astype(int), " (fundamental theorem of topology: boundary of a boundary = 0, Problem Sheet 2)")

L1 = SC.hodge_laplacian_matrix(rank=1).toarray()
L1_down = SC.down_laplacian_matrix(rank=1).toarray()   # B1^T B1
L1_up = SC.up_laplacian_matrix(rank=1).toarray()       # B2 B2^T
print("L1 == L1_down + L1_up ?", np.allclose(L1, L1_down + L1_up))


### A simplicial message-passing layer, by hand

Mirroring the graph message-passing formula, but now it's not vertices communicating with each other but **simplices of contiguous dimensions**. A simplex has two
kinds of neighbors: 
1. its **boundary** faces (one dimension down, via $B_k$) 
2. and its **upper-adjacent** simplices (same dimension, sharing a coface, via $L_k^{\text{up}}$) 

These are exactly the two neighborhood relations that we said were enough for the Simplicial WL update. We build a small edge-level ($k=1$) layer that aggregates both.


In [ ]:
class SimplicialConv(nn.Module):
    # One message-passing layer for 1-dimensional simplices (edges), combining:
    #  (a) a BOUNDARY message from the 2 endpoint vertices (via B1), and
    #  (b) an UPPER-ADJACENCY message from edges sharing a triangle (via A1_up).
    # Compare with h_i = phi(x_i, +_{j in N_i} psi(x_j)) of slide 30.
    def __init__(self, dim0, dim1, hidden):
        super().__init__()
        self.lin_self  = nn.Linear(dim1, hidden)
        self.lin_bound = nn.Linear(dim0, hidden)
        self.lin_up    = nn.Linear(dim1, hidden)

    def forward(self, x0, x1, B1, A1_up):
        # TODO(7): build the two neighborhood messages.
        #   boundary_msg: sum of the 2 endpoint-vertex features of each edge -- use B1 and the node features x0
        #   upper_msg:    sum over upper-adjacent edges (sharing a triangle) -- use A1_up and the edge features x1
        boundary_msg = ...
        upper_msg = ...
        h = self.lin_self(x1) + self.lin_bound(boundary_msg) + self.lin_up(upper_msg)
        return F.relu(h)

x0 = torch.eye(n0)                                                  # one-hot vertex features
x1 = torch.ones(n1, 1)                                               # constant edge features
B1_t = torch.tensor(B1, dtype=torch.float32)
A1_up = torch.tensor(SC.adjacency_matrix(rank=1).toarray(), dtype=torch.float32)

torch.manual_seed(0)
layer = SimplicialConv(dim0=n0, dim1=1, hidden=6)
h1 = layer(x0, x1, B1_t, A1_up)
print("updated edge features, shape", h1.shape, ":\n", h1)


### Optional: the official library layer (TopoModelX)

`TopoNetX` builds the topological domain and its matrices; the actual learnable layers of
the TopoX ecosystem live in **TopoModelX**. 
Here is the same idea, using its
`Conv` layer directly on the neighborhood matrix TopoNetX just computed for us.


In [ ]:
try:
    !pip install -q topomodelx
    from topomodelx.base.conv import Conv
    from topomodelx.utils.sparse import from_sparse

    A1_up_sparse = from_sparse(SC.adjacency_matrix(rank=1))
    conv = Conv(in_channels=1, out_channels=6, update_func="relu")
    out = conv(x1, A1_up_sparse)
    print("TopoModelX Conv output, shape", out.shape, ":\n", out)
except Exception as e:
    print("topomodelx not available in this environment, skipping (the hand-rolled layer above covers the same idea):", e)